# Mixed-precision evaluations with DMDS Hamiltonian

This notebook mirrors **evaluations.ipynb** (constraints, evaluator, filemaker workflow) but uses the **DMDS** (dimethyl disulfide) electronic Hamiltonian as in the mixed-precision tutorial. The constraint is the **spacetime cost** of the GPU emulator (peak memory from CostAnalyzer, counting precisions).

1. Build DMDS qubit Hamiltonian
2. Define constraints (EmulatorMemory = spacetime cost)
3. Test generator and exact evolution
4. Generate initial program and evaluator with filemaker
5. Test evaluator and (optionally) run OpenEvolve

In [ ]:
import numpy as np
import scipy
import trovolut as trv
trv.set_backend('gpu-mp-emulator')

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

## 1. DMDS Hamiltonian

Build the qubit Hamiltonian for dimethyl disulfide (CSSC) with a minimal active space, same as in the mixed-precision tutorial.

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
from trovolut import qm

# Generate DMDS molecule
mol = Chem.AddHs(Chem.MolFromSmiles("CSSC"))
AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
AllChem.UFFOptimizeMolecule(mol)

conf = mol.GetConformer()
atoms = [a.GetSymbol() for a in mol.GetAtoms()]
xyz = (
    f"{len(atoms)}\\ndmds generated by RDKit\\n"
    + "\\n".join(
        f"{s} {conf.GetAtomPosition(i).x:.8f} "
        f"{conf.GetAtomPosition(i).y:.8f} "
        f"{conf.GetAtomPosition(i).z:.8f}"
        for i, s in enumerate(atoms)
    )
)

H_fermion = qm.ElectronicHamiltonian.from_xyz(
    xyz_source=xyz,
    basis="sto-3g",
    charge=0,
    spin_mult=1,
    use_gpu4pyscf=False,
    freeze_core=True,
    n_core_keep=1,
    n_virt_keep=1,
)

H_qubit = H_fermion.to_qubits(mapping="jordan-wigner", sparse=True)
qmeta = H_qubit.decompose()
n_qubits = qmeta['n_qubits']
n_terms = len(qmeta['terms'])

def get_hamiltonian():
    """Return the DMDS qubit Hamiltonian (for test cases and cost analysis)."""
    return H_qubit

print(f"DMDS: {n_qubits} qubits, Hilbert dim {2**n_qubits}, {n_terms} Pauli terms")

## 2. Define constraints

Constraint: **spacetime cost** of the dynamics method (peak memory in MB from CostAnalyzer, which counts per-term precisions). We also keep a small error tolerance so evolution remains accurate.

In [ ]:
from trovolut.core.base_constraints import Time, Error, Group
from trovolut.backends.gpu_mp_emulator.constraints import EmulatorMemory

# Spacetime cost: peak memory (MB) for one dynamics method — lower is better
memory_constr = EmulatorMemory(max_mb=50.0, weight=1.0, hard=False, score_fn="linear")
error_constr = Error(max_error=0.1, hard=True, hard_penalty=-100.0)

print(f"Constraint 1: {memory_constr}\n\t{memory_constr.formula()}")
print(f"Constraint 2: {error_constr}\n\t{error_constr.formula()}")

group_constr = Group([memory_constr, error_constr])
print(f"\n{group_constr.formula()}")

## 3. Test generator and exact evolution

In [ ]:
def get_test_cases():
    """Return (hamiltonian, initial_state, time). Hamiltonian is H_qubit (supports .terms())."""
    hamiltonian = get_hamiltonian()
    n = 2 ** n_qubits
    np.random.seed(42)
    psi0 = np.random.randn(n) + 1j * np.random.randn(n)
    psi0 /= np.linalg.norm(psi0)
    t = 0.1
    return hamiltonian, psi0, t

def exact_evolution(hamiltonian, t, psi0):
    """Reference solution for error measurement."""
    if hasattr(hamiltonian, 'to_matrix'):
        H = hamiltonian.to_matrix()
    elif isinstance(hamiltonian, (list, tuple)):
        H = sum(hamiltonian)
    else:
        H = np.asarray(hamiltonian)
    if hasattr(H, 'toarray'):
        H = H.toarray()
    return scipy.linalg.expm(-1j * H * t) @ psi0

## 4. Generate initial program and evaluator

The program exposes **`evolution`** (MPTrotterEvolution) and **`run_dynamics`**. The evaluator uses `evolution` to compute spacetime cost (peak_memory_bytes) via CostAnalyzer so the EmulatorMemory constraint can score it.

In [ ]:
from trovolut.backends.gpu_mp_emulator.dynamics import MPTrotterEvolution

Path("opt").mkdir(exist_ok=True)

# Emit this block into initial_program.py so evaluator can read module.evolution for spacetime cost
evolution_and_schedule = '''
DEFAULT_PRECISION_SCHEDULE = ["float32", "float16", "float8_e4m3"]
from trovolut.backends.gpu_mp_emulator.dynamics import MPTrotterEvolution
evolution = MPTrotterEvolution(trotter_steps=20, suzuki_order=2, default_precision="float32", accumulate_precision="float64")
'''

def run_dynamics(hamiltonian, t, psi0):
    """Mixed-precision Trotter evolution with per-term precision schedule."""
    return evolution.evolve(
        hamiltonian,
        psi0,
        t,
        precision_schedule=DEFAULT_PRECISION_SCHEDULE,
        decomposition="exact",
    )

workflow = trv.filemaker.Workflow(
    [trv.filemaker.EVO_START, evolution_and_schedule, run_dynamics, trv.filemaker.EVO_END],
    out_path="opt/initial_program.py",
)

evaluator_gen = trv.filemaker.Evaluator(
    test_generator=get_test_cases,
    workflow=workflow,
    constraints=group_constr,
    exact_fn=exact_evolution,
    backend="gpu-mp-emulator",
    out_path="opt/evaluator.py",
    n_trials=5,
)

init_path = workflow.build()
evaluator_path = evaluator_gen.build()

print("Generated files:")
print(f"  Initial program: {init_path}")
print(f"  Evaluator:       {evaluator_path}")

## 5. Test evaluator

In [ ]:
exec(open(evaluator_path).read())
result = evaluator(str(init_path))

print("OpenEvolve-style output:")
print(f"  combined_score: {result['combined_score']:.6f}")
print(f"  std_score:      {result.get('std_score', 0.0):.6f}")
print(f"  all_satisfied:  {result['all_satisfied']}")
print(f"\nMetrics:")
print(f"  time:               {result.get('time', 0):.6f}")
print(f"  error:              {result.get('error', 0):.6e}")
print(f"  memory_peak_bytes:  {result.get('memory_peak_bytes', 0):.0f}")
print(f"\nConstraint details:")
for name, info in result.get('details', {}).items():
    print(f"  {name}: value={info.get('value')}, score={info.get('score')}, satisfied={info.get('satisfied')}")

## 6. LLM instructions (for OpenEvolve)

In [ ]:
print(group_constr.generate_llm_instructions())